In [ ]:
import pandas as pd
import numpy as np

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore",category=ConvergenceWarning)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN

In [ ]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';', engine='python',on_bad_lines='skip')

In [ ]:
df.drop('duration', axis=1, inplace=True)

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [ ]:
df['y'] = df['y'].map({'no': 0, 'yes': 1})

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop('y', axis=1)
y = df['y']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

In [ ]:
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train, y_train)

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=20 ,
        random_state=42,
        min_samples_split=10
        ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        max_depth=20
        ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=300,
        random_state=42,
        alpha=0.0001,
        activation='relu',
        solver='adam',
        early_stopping=True
        ),
    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        learning_rate=0.2
        )
}

In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
result = []

In [ ]:
def evaluate_models(X, y, label):
    print(f"\n===== {label} =====")

    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        print(f"\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")
        ac  = round(np.mean(scores['test_accuracy']),4)
        pc  = round(np.mean(scores['test_precision']),4)
        rec = round(np.mean(scores['test_recall']),4)
        f1  = round(np.mean(scores['test_f1']),4)
        roc = round(np.mean(scores['test_roc_auc']),4)
        result.append([name , ac , pc , rec ,f1 , roc])


In [ ]:
evaluate_models(X_train, y_train, "Baseline (No Sampling)")


===== Baseline (No Sampling) =====

Model: Decision Tree
Accuracy:  0.8779
Precision: 0.4365
Recall:    0.2928
F1 Score:  0.3500
ROC-AUC:   0.6758

Model: Random Forest
Accuracy:  0.8951
Precision: 0.5764
Recall:    0.2662
F1 Score:  0.3635
ROC-AUC:   0.7832

Model: MLP
Accuracy:  0.8980
Precision: 0.6460
Recall:    0.2144
F1 Score:  0.3208
ROC-AUC:   0.7834

Model: XGBoost
Accuracy:  0.8981
Precision: 0.6091
Recall:    0.2686
F1 Score:  0.3720
ROC-AUC:   0.7917


In [ ]:
evaluate_models(X_train_sm, y_train_sm, "SMOTE")


===== SMOTE =====

Model: Decision Tree
Accuracy:  0.9108
Precision: 0.9320
Recall:    0.8863
F1 Score:  0.9086
ROC-AUC:   0.9378

Model: Random Forest
Accuracy:  0.9342
Precision: 0.9457
Recall:    0.9214
F1 Score:  0.9334
ROC-AUC:   0.9791

Model: MLP
Accuracy:  0.8824
Precision: 0.8841
Recall:    0.8804
F1 Score:  0.8822
ROC-AUC:   0.9487

Model: XGBoost
Accuracy:  0.9366
Precision: 0.9678
Recall:    0.9033
F1 Score:  0.9345
ROC-AUC:   0.9741


In [ ]:
evaluate_models(X_train_ad, y_train_ad, "ADASYN")


===== ADASYN =====

Model: Decision Tree
Accuracy:  0.9078
Precision: 0.9268
Recall:    0.8855
F1 Score:  0.9056
ROC-AUC:   0.9373

Model: Random Forest
Accuracy:  0.9364
Precision: 0.9449
Recall:    0.9266
F1 Score:  0.9357
ROC-AUC:   0.9801

Model: MLP
Accuracy:  0.8723
Precision: 0.8701
Recall:    0.8751
F1 Score:  0.8725
ROC-AUC:   0.9424

Model: XGBoost
Accuracy:  0.9353
Precision: 0.9699
Recall:    0.8983
F1 Score:  0.9327
ROC-AUC:   0.9740


In [ ]:
df_result= pd.DataFrame(result , columns=["Model" , "Accuracy" , "Precision" , "Recall","F1 Score" , "ROC-AUC"])

In [ ]:
df_result.to_excel("Model_res.xlsx" , index=False)

In [ ]:
from google.colab import files
files.download("Model_res.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')